# Bloque 2: Regularización y Estrategias de Modelamiento
## Tema 1: L1, L2 y Elastic Net

**Autor:** Sofia Dextre | Hack with DSC PUCP  
**Nivel:** Avanzado  
**Duración estimada:** 45 minutos

---

### 🎯 Objetivo
Entender cómo las técnicas de regularización (L1, L2, Elastic Net) controlan el sobreajuste eligiendo qué tipo de soluciones penalizar, y cuándo usar cada una según las características de tus datos.

### 📚 Temas a cubrir
1. ¿Qué es regularizar y por qué importa?
2. Regularización L1 (Lasso): selección de variables
3. Regularización L2 (Ridge): estabilidad
4. Elastic Net: lo mejor de ambos mundos
5. Aplicación práctica: Credit Scoring
6. Comparación y decisiones

---

## 0. Setup y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Librerías importadas")

## 1. Concepto: ¿Qué es regularizar?

### El dilema fundamental

**Sin regularización:**
- El modelo busca minimizar solo el error en los datos de entrenamiento
- Puede memorizar patrones específicos de esos datos
- Generaliza mal a datos nuevos (SOBREAJUSTE)

**Con regularización:**
- El modelo minimiza error EN LOS DATOS + un costo por complejidad
- Castiga soluciones complicadas o con muchos coeficientes grandes
- Generaliza mejor

### Fórmula general

$$\text{Objetivo} = \text{Loss} + \lambda \cdot \text{Penalización}(\beta)$$

Donde:
- **Loss:** Error en los datos (MSE, Logloss, etc.)
- **λ (Lambda):** Intensidad de regularización (0 = sin castigo, ∞ = castigo extremo)
- **Penalización:** Estructura que define qué castigamos (L1, L2, o mezcla)

### Visualización: ¿Qué pasa sin regularización?

In [ ]:
# Crear datos sintéticos: regresión logística con sobreajuste
np.random.seed(42)
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Entrenar modelos con diferentes λ
lambdas = [0.001, 0.01, 0.1, 1, 10, 100]
train_scores = []
test_scores = []

for lam in lambdas:
    model = LogisticRegression(C=1/lam, max_iter=1000)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Curva de regularización
axes[0].plot(np.log10(lambdas), train_scores, 'o-', linewidth=2, markersize=8, label='Train', color='#2E86AB')
axes[0].plot(np.log10(lambdas), test_scores, 's-', linewidth=2, markersize=8, label='Test', color='#A23B72')
axes[0].axvline(np.log10(0.1), color='red', linestyle='--', alpha=0.5, label='λ óptimo')
axes[0].set_xlabel('log₁₀(λ)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Trade-off: Ajuste vs Generalización', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Diferencia (overfitting)
overfitting = np.array(train_scores) - np.array(test_scores)
colors = ['#FF6B6B' if x > 0.1 else '#51CF66' for x in overfitting]
axes[1].bar(range(len(lambdas)), overfitting, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_xlabel('Intensidad de regularización (λ)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Sobreajuste (Train - Test)', fontsize=12, fontweight='bold')
axes[1].set_title('Reducción del Sobreajuste', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(len(lambdas)))
axes[1].set_xticklabels([f'{l:.3f}' for l in lambdas], rotation=45)
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("📊 Interpretación:")
print("  • Sin regularización (λ pequeño): Train≈1.0, Test≈0.7 → SOBREAJUSTE")
print("  • Con regularización óptima: Train≈0.85, Test≈0.85 → EQUILIBRIO")
print("  • Regularización excesiva (λ grande): Train≈0.8, Test≈0.8 → SUBAJUSTE")

---
## 2. Regularización L1 (Lasso): "Parsimonia"

### ¿Qué castiga L1?

$$\text{Loss} + \lambda \sum_{j=1}^{p} |\beta_j|$$

**Características geométricas:**
- La restricción forma un **diamante** en 2D (rombo)
- Las **esquinas del diamante** alinean con los ejes (donde βⱼ = 0)
- La solución optimal tiende a caer en una esquina
- **Resultado:** Coeficientes se ponen EXACTAMENTE a cero

### ¿Qué consigues?

✓ **Selección implícita de variables:** Algunos coeficientes → 0 exactamente  
✓ **Modelo parsimonioso:** Menos variables activas  
✓ **Interpretabilidad:** Solo variables importantes en la ecuación  
✓ **Trazabilidad:** Fácil de auditar y explicar

### Visualización: Geometría L1 vs L2

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Crear grid
beta1 = np.linspace(-2, 2, 100)
beta2 = np.linspace(-2, 2, 100)
B1, B2 = np.meshgrid(beta1, beta2)

# Contornos Loss (elipse)
loss = (B1 - 0.5)**2 + (B2 - 0.5)**2

# ===== LASSO (L1) ====="
ax = axes[0]
l1_norm = np.abs(B1) + np.abs(B2)  # Diamante
ax.contour(B1, B2, loss, levels=8, colors='gray', alpha=0.5, linewidths=1)
ax.contour(B1, B2, l1_norm, levels=[0.5, 1.0, 1.5, 2.0], colors='#FF6B6B', linewidths=2.5)
ax.plot(0.5, 0.5, 'g*', markersize=20, label='Óptimo (sin penalización)')
ax.plot(1, 0, 'ro', markersize=10, label='Solución L1', markerfacecolor='#FF6B6B', markeredgewidth=2)
ax.set_xlabel('β₁', fontsize=12, fontweight='bold')
ax.set_ylabel('β₂', fontsize=12, fontweight='bold')
ax.set_title('Lasso (L1): Diamante\nFavored edges → β=0', fontsize=12, fontweight='bold', color='#FF6B6B')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)

# ===== RIDGE (L2) ====="
ax = axes[1]
l2_norm = B1**2 + B2**2  # Círculo
ax.contour(B1, B2, loss, levels=8, colors='gray', alpha=0.5, linewidths=1)
ax.contour(B1, B2, l2_norm, levels=[0.5, 1.0, 1.5, 2.0], colors='#4ECDC4', linewidths=2.5)
ax.plot(0.5, 0.5, 'g*', markersize=20, label='Óptimo (sin penalización)')
ax.plot(0.35, 0.35, 'bo', markersize=10, label='Solución L2', markerfacecolor='#4ECDC4', markeredgewidth=2)
ax.set_xlabel('β₁', fontsize=12, fontweight='bold')
ax.set_ylabel('β₂', fontsize=12, fontweight='bold')
ax.set_title('Ridge (L2): Círculo\nFavored smooth → β≠0', fontsize=12, fontweight='bold', color='#4ECDC4')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)

# ===== ELASTIC NET (L1+L2) ====="
ax = axes[2]
elastic_norm = 0.5 * (l1_norm + l2_norm)  # Mezcla
ax.contour(B1, B2, loss, levels=8, colors='gray', alpha=0.5, linewidths=1)
ax.contour(B1, B2, elastic_norm, levels=[0.5, 1.0, 1.5, 2.0], colors='#F38181', linewidths=2.5)
ax.plot(0.5, 0.5, 'g*', markersize=20, label='Óptimo (sin penalización)')
ax.plot(0.6, 0.2, 'mo', markersize=10, label='Solución Elastic Net', markerfacecolor='#F38181', markeredgewidth=2)
ax.set_xlabel('β₁', fontsize=12, fontweight='bold')
ax.set_ylabel('β₂', fontsize=12, fontweight='bold')
ax.set_title('Elastic Net (L1+L2)\nHybrid → mix of both', fontsize=12, fontweight='bold', color='#F38181')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)

plt.suptitle('Geometría de Regularización: Por qué L1 selecciona variables', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("🔍 Intuición geométrica:")
print("  L1 (diamante): Las esquinas están en los ejes → βⱼ=0 con alta probabilidad")
print("  L2 (círculo): Es suave en todas direcciones → βⱼ reducidos pero ≠0")
print("  EN: Combina lo mejor de ambos")

### Práctica: Lasso en acción

In [ ]:
# Crear dataset con muchas variables
np.random.seed(42)
n_samples, n_features = 200, 50
X, y = make_classification(n_samples=n_samples, n_features=n_features, n_informative=10, 
                            n_redundant=20, random_state=42)
X = StandardScaler().fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar Lasso con diferentes λ
lambdas = np.logspace(-3, 0, 50)
n_coefs_active = []
train_scores = []
test_scores = []

for lam in lambdas:
    model = LogisticRegression(penalty='l1', solver='liblinear', C=1/lam, max_iter=1000)
    model.fit(X_train, y_train)
    
    # Contar coeficientes no-cero
    n_coefs_active.append(np.sum(model.coef_ != 0))
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variables seleccionadas
ax = axes[0]
ax.plot(np.log10(lambdas), n_coefs_active, 'o-', linewidth=2.5, markersize=6, color='#FF6B6B')
ax.fill_between(np.log10(lambdas), n_coefs_active, alpha=0.3, color='#FF6B6B')
ax.axhline(10, color='green', linestyle='--', alpha=0.7, linewidth=2, label='Variables informativas')
ax.set_xlabel('log₁₀(λ)', fontsize=12, fontweight='bold')
ax.set_ylabel('# Coeficientes activos (≠0)', fontsize=12, fontweight='bold')
ax.set_title('Lasso: Selección de Variables', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Rendimiento
ax = axes[1]
ax.plot(np.log10(lambdas), train_scores, 'o-', linewidth=2, markersize=6, label='Train', color='#2E86AB')
ax.plot(np.log10(lambdas), test_scores, 's-', linewidth=2, markersize=6, label='Test', color='#A23B72')
ax.set_xlabel('log₁₀(λ)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Lasso: Rendimiento Train vs Test', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Lasso en acción:")
print(f"  • Con λ=0.001: {n_coefs_active[0]} variables activas")
print(f"  • Con λ=0.1: {n_coefs_active[25]} variables activas (cercano al óptimo)")
print(f"  • Con λ=1.0: {n_coefs_active[-1]} variables activas")
print(f"\n  ✨ Lasso encontró ~{n_coefs_active[25]} variables importantes de un total de 50")

---
## 3. Regularización L2 (Ridge): "Estabilidad"

### ¿Qué castiga L2?

$$\text{Loss} + \lambda \sum_{j=1}^{p} \beta_j^2$$

**Características geométricas:**
- La restricción forma un **círculo** en 2D
- El círculo es **suave** en todas direcciones
- La solución optimal reduce todos los coeficientes proporcionalmente
- **Resultado:** Coeficientes se reducen, pero rara vez llegan a cero exactamente

### ¿Qué consigues?

✓ **Coeficientes controlados:** Más pequeños pero presentes  
✓ **Estabilidad ante multicolinealidad:** Distribuye el peso entre variables correlacionadas  
✓ **Cambios suaves:** Pequeños cambios en datos → pequeños cambios en coeficientes  
✓ **Todas las variables participan:** No elimina completamente ninguna

### Práctica: Ridge vs Lasso ante multicolinealidad

In [ ]:
# Crear datos con multicolinealidad
np.random.seed(42)
n_samples = 200

# Variable base
x1 = np.random.randn(n_samples)
# Variables fuertemente correlacionadas
x2 = x1 + np.random.randn(n_samples) * 0.1  # correlación ≈ 0.99
x3 = x1 + np.random.randn(n_samples) * 0.1

X = np.column_stack([x1, x2, x3])
y = x1 + np.random.randn(n_samples) * 0.5

# Escalar
X = StandardScaler().fit_transform(X)
y = (y - y.mean()) / y.std()

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Ridge
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)
ridge_coefs = ridge.coef_

# Lasso
lasso = Lasso(alpha=0.01, max_iter=2000)
lasso.fit(X_train, y_train)
lasso_coefs = lasso.coef_

# Sin regularización
ols = Ridge(alpha=0)  # OLS
ols.fit(X_train, y_train)
ols_coefs = ols.coef_

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(3)
width = 0.25

ax.bar(x_pos - width, ols_coefs, width, label='OLS (sin regularización)', color='#FF6B6B', alpha=0.8)
ax.bar(x_pos, ridge_coefs, width, label='Ridge (L2)', color='#4ECDC4', alpha=0.8)
ax.bar(x_pos + width, lasso_coefs, width, label='Lasso (L1)', color='#FFE66D', alpha=0.8)

ax.set_ylabel('Coeficiente', fontsize=12, fontweight='bold')
ax.set_xlabel('Variable', fontsize=12, fontweight='bold')
ax.set_title('Ridge vs Lasso ante Multicolinealidad', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['x1', 'x2 (correlada)', 'x3 (correlada)'])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n📊 Análisis de Multicolinealidad:")
print(f"\nOLS (sin regularización):")
print(f"  Coeficientes: {ols_coefs}")
print(f"  ⚠️  Muy grandes y volatiles (inestables)")
print(f"\nRidge (L2):")
print(f"  Coeficientes: {ridge_coefs}")
print(f"  ✓ Distribuye el peso entre variables correlacionadas")
print(f"\nLasso (L1):")
print(f"  Coeficientes: {lasso_coefs}")
print(f"  ✓ Mantiene solo una variable, elimina las otras")

---
## 4. Elastic Net: Lo mejor de ambos mundos

### ¿Qué castiga Elastic Net?

$$\text{Loss} + \lambda \left[\alpha \sum_{j=1}^{p} |\beta_j| + (1-\alpha) \sum_{j=1}^{p} \beta_j^2 \right]$$

**Parámetros:**
- **λ:** Intensidad total de regularización
- **α:** Mezcla (0 = Ridge puro, 1 = Lasso puro)

### ¿Qué consigues?

✓ **Selección de variables (L1):** Algunos coeficientes → 0  
✓ **Estabilidad (L2):** Coeficientes controlados ante multicolinealidad  
✓ **Flexibilidad:** Ajusta α según tus datos

### Práctica: Elastic Net con búsqueda de α

In [ ]:
# Usar los datos del ejemplo anterior (multicolinealidad)

# Entrenar Elastic Net con diferentes α
alphas_mix = np.linspace(0, 1, 11)  # 0=Ridge, 1=Lasso
n_coefs_active_en = []
train_scores_en = []
test_scores_en = []

for alpha_mix in alphas_mix:
    en = ElasticNet(alpha=0.05, l1_ratio=alpha_mix, max_iter=2000)
    en.fit(X_train, y_train)
    
    n_coefs_active_en.append(np.sum(en.coef_ != 0))
    train_scores_en.append(en.score(X_train, y_train))
    test_scores_en.append(en.score(X_test, y_test))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variables seleccionadas
ax = axes[0]
colors_alpha = ['#4ECDC4' if a < 0.5 else ('#F38181' if a > 0.5 else '#95E1D3') for a in alphas_mix]
ax.bar(range(len(alphas_mix)), n_coefs_active_en, color=colors_alpha, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.7, linewidth=2, label='50/50 (L1=L2)')
ax.set_xlabel('α (L1 ratio)', fontsize=12, fontweight='bold')
ax.set_ylabel('# Coeficientes activos', fontsize=12, fontweight='bold')
ax.set_title('Elastic Net: Impacto de α', fontsize=13, fontweight='bold')
ax.set_xticks(range(len(alphas_mix)))
ax.set_xticklabels([f'{a:.1f}' for a in alphas_mix])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Rendimiento
ax = axes[1]
ax.plot(alphas_mix, train_scores_en, 'o-', linewidth=2.5, markersize=8, label='Train', color='#2E86AB')
ax.plot(alphas_mix, test_scores_en, 's-', linewidth=2.5, markersize=8, label='Test', color='#A23B72')
ax.fill_between(alphas_mix, test_scores_en, alpha=0.2, color='#A23B72')
ax.set_xlabel('α (L1 ratio)', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Elastic Net: Rendimiento según α', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_alpha_idx = np.argmax(test_scores_en)
best_alpha = alphas_mix[best_alpha_idx]

print(f"\n✓ Elastic Net encontró α óptimo: {best_alpha:.1f}")
print(f"  • α=0.0 (Ridge puro): {n_coefs_active_en[0]} variables")
print(f"  • α={best_alpha:.1f} (óptimo): {n_coefs_active_en[best_alpha_idx]} variables")
print(f"  • α=1.0 (Lasso puro): {n_coefs_active_en[-1]} variables")

---
## 5. Aplicación Real: Credit Scoring

### Caso: Predicción de Churn con Regularización

En credit scoring necesitamos:
1. **Precisión:** Detectar clientes en riesgo
2. **Interpretabilidad:** Explicar por qué rechazamos/aceptamos
3. **Trazabilidad:** Auditar y monitorear

¿Cuál regularización elegir?

In [ ]:
# Crear dataset de credit scoring simulado
np.random.seed(42)
n_samples = 1000

# Features
credit_score = np.random.uniform(300, 850, n_samples)
annual_income = np.random.uniform(20000, 200000, n_samples)
credit_utilization = np.random.uniform(0, 100, n_samples)
payment_history = np.random.choice([0, 1], n_samples, p=[0.3, 0.7])
debt_to_income = np.random.uniform(0, 1, n_samples)

# Features con ruido
noise_features = np.random.randn(n_samples, 15)

# Target (basado en features principales)
y = (credit_score > 650).astype(int) * 0.7 + \
    (annual_income > 60000).astype(int) * 0.2 + \
    (credit_utilization < 30).astype(int) * 0.1
y = (y > 0.5).astype(int)
y = np.where(np.random.rand(n_samples) < 0.15, 1 - y, y)  # Flip some labels

# Construir X
X = np.column_stack([
    credit_score,
    annual_income,
    credit_utilization,
    payment_history,
    debt_to_income,
    noise_features
])

# Escalar
X = StandardScaler().fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test")
print(f"Features: 5 informativas + 15 ruido = 20 totales")
print(f"Target balance: {y.mean():.1%} default")

### Comparación: Sin regularización vs L1 vs L2 vs Elastic Net

In [ ]:
# Entrenar todos los modelos
models = {
    'Sin regularización': LogisticRegression(penalty=None, max_iter=1000),
    'L1 (Lasso)': LogisticRegression(penalty='l1', solver='liblinear', C=0.1),
    'L2 (Ridge)': LogisticRegression(penalty='l2', C=0.1),
    'Elastic Net (α=0.5)': LogisticRegression(penalty='elasticnet', solver='saga', C=0.1, l1_ratio=0.5, max_iter=1000)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    
    # Métricas
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    n_active = np.sum(model.coef_ != 0)
    
    results[name] = {
        'train_auc': train_auc,
        'test_auc': test_auc,
        'n_active': n_active,
        'model': model,
        'coef': model.coef_[0]
    }

# Tabla comparativa
comparison_df = pd.DataFrame({
    'Modelo': list(results.keys()),
    'Train AUC': [results[m]['train_auc'] for m in results],
    'Test AUC': [results[m]['test_auc'] for m in results],
    'Variables activas': [results[m]['n_active'] for m in results],
    'Sobreajuste': [results[m]['train_auc'] - results[m]['test_auc'] for m in results]
})

print("\n" + "="*80)
print("COMPARACIÓN: Regularización en Credit Scoring")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualizar
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# AUC
ax = axes[0, 0]
models_list = list(results.keys())
train_aucs = [results[m]['train_auc'] for m in models_list]
test_aucs = [results[m]['test_auc'] for m in models_list]
x_pos = np.arange(len(models_list))
width = 0.35
ax.bar(x_pos - width/2, train_aucs, width, label='Train', color='#2E86AB', alpha=0.8)
ax.bar(x_pos + width/2, test_aucs, width, label='Test', color='#A23B72', alpha=0.8)
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Rendimiento Predictivo', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.5, 1.0])

# Variables activas
ax = axes[0, 1]
n_actives = [results[m]['n_active'] for m in models_list]
colors = ['#FF6B6B' if n > 15 else '#51CF66' for n in n_actives]
ax.bar(x_pos, n_actives, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.axhline(5, color='green', linestyle='--', alpha=0.7, linewidth=2, label='Variables informativas')
ax.set_ylabel('# Variables', fontsize=11, fontweight='bold')
ax.set_title('Parsimonia del Modelo', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Sobreajuste
ax = axes[1, 0]
overfits = [results[m]['train_auc'] - results[m]['test_auc'] for m in models_list]
colors = ['#FF6B6B' if x > 0.05 else '#51CF66' for x in overfits]
ax.bar(x_pos, overfits, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Train AUC - Test AUC', fontsize=11, fontweight='bold')
ax.set_title('Sobreajuste (Gap Train-Test)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, rotation=15, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

# Coeficientes top-5
ax = axes[1, 1]
best_model_name = comparison_df.loc[comparison_df['Test AUC'].idxmax(), 'Modelo']
best_coef = results[best_model_name]['coef']
top_indices = np.argsort(np.abs(best_coef))[-10:]
feature_names = [f'credit_score', 'annual_income', 'credit_util', 'payment_hist', 'debt_income'] + [f'noise_{i}' for i in range(15)]
top_features = [feature_names[i] for i in top_indices]
top_coefs = best_coef[top_indices]
colors_coef = ['#2E86AB' if c > 0 else '#FF6B6B' for c in top_coefs]
ax.barh(range(len(top_features)), top_coefs, color=colors_coef, alpha=0.8, edgecolor='black', linewidth=1)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=9)
ax.set_xlabel('Coeficiente', fontsize=11, fontweight='bold')
ax.set_title(f'Top-10 Features ({best_model_name})', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### 📊 Conclusiones: Cuándo usar cada regularización

In [ ]:
print("\n" + "="*80)
print("RECOMENDACIONES PRÁCTICAS: Cuándo usar L1, L2, Elastic Net")
print("="*80)

recommendations = pd.DataFrame({
    'Regularización': ['Sin (OLS/LR)', 'L1 (Lasso)', 'L2 (Ridge)', 'Elastic Net'],
    'Cuándo usar': [
        'Pocos features, datos limpios, bajo riesgo overfitting',
        'Muchas variables, necesitas seleccionar las mejores, explicabilidad crucial',
        'Multicolinealidad, stabilidad importante, puedes usar todas las variables',
        'Muchas variables + multicolinealidad, quieres algo equilibrado'
    ],
    'Ventaja principal': [
        'Sin penalización extra',
        'Selecciona variables (β=0)',
        'Distribuye peso entre correlacionadas',
        'Combina ambas ventajas'
    ],
    'Desventaja': [
        'Propenso a overfitting',
        'Puede ser inestable con multicolinealidad',
        'Mantiene todas las variables',
        'Requiere ajustar α'
    ]
})

print(recommendations.to_string(index=False))
print("="*80)

print("\n" + "⭐"*40)
print("\nPARA CREDIT SCORING (nuestro caso):")
print("\n✓ L1 es mejor si:")
print("  • Necesitas un modelo muy interpretable (regulador lo exige)")
print("  • Pocos clientes en default (baja información)")
print("  • Variables ruidosas que quieres eliminar")

print("\n✓ L2 es mejor si:")
print("  • Variables financieras correlacionadas (tasa, interés, etc.)")
print("  • Quieres estabilidad en el monitoreo")
print("  • Datos abundantes")

print("\n✓ Elastic Net es mejor si:")
print("  • Tienes lo mejor de ambos mundos: muchas variables + multicolinealidad")
print("  • Quieres experimentar con α")
print("  • Necesitas un punto medio entre parsimonia y estabilidad")

print("\n" + "="*80)

---
## 6. Ejercicio: Elige la regularización

Tienes 4 escenarios. Para cada uno, ¿cuál regularización elegirías y por qué?

In [ ]:
# Escenarios para practicar
scenarios = [
    {
        'nombre': 'Escenario 1: Marketing Digital',
        'descripcion': '1000 clientes, 200 features (clicks, impresiones, conversiones), bajo overfitting',
        'respuesta': '→ Sin regularización o L2 (pocas características ruidosas)',
        'justificacion': 'Datos limpios, necesitamos usar toda la información disponible'
    },
    {
        'nombre': 'Escenario 2: Genómica',
        'descripcion': '100 muestras, 20,000 genes, necesitas identificar los principales',
        'respuesta': '→ L1 (Lasso)',
        'justificacion': 'p >> n, necesitas seleccionar genes importantes, sparsity es crucial'
    },
    {
        'nombre': 'Escenario 3: Economía (índices correlacionados)',
        'descripcion': '1000 meses, 50 features (tasas, índices, precios). Muchas correlacionadas',
        'respuesta': '→ L2 (Ridge) o Elastic Net',
        'justificacion': 'Multicolinealidad natural, L2 distribuye el peso, Ridge es más estable'
    },
    {
        'nombre': 'Escenario 4: Fraude (muchas features + multicolinealidad)',
        'descripcion': '500K transacciones, 150 features (device, geolocalización, montos, tiempos)',
        'respuesta': '→ Elastic Net (α=0.5-0.7)',
        'justificacion': 'Necesitas seleccionar (L1) pero con estabilidad ante correlación (L2)'
    }
]

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{i}. {scenario['nombre']}")
    print(f"   Contexto: {scenario['descripcion']}")
    print(f"   {scenario['respuesta']}")
    print(f"   Justificación: {scenario['justificacion']}")

print("\n" + "="*80)

---
## 📝 Resumen: Tabla de Decisión Rápida

| Situación | Usa | Por qué |
|-----------|-----|----------|
| **Pocos features, datos limpios** | Sin regularización | No hay riesgo de overfitting |
| **Muchas features, necesitas seleccionar** | **L1** | Coeficientes → 0, parsimonia |
| **Multicolinealidad fuerte** | **L2** | Coeficientes controlados, estables |
| **Muchas features + multicolinealidad** | **Elastic Net** | Selección + estabilidad |
| **Explicabilidad es crítica (reguladores)** | **L1** | Menos variables → más fácil de justificar |
| **Datos pequeños (p > n)** | **L1** | Necesita selecting, evita overfitting extremo |
| **Datos grandes, mucho ruido** | **L2 o Elastic Net** | Distribuye complejidad |
| **Incertidumbre: no sabes cuál** | **Elastic Net** | Experimenta con α, es flexible |

---

## ✅ Checkpoints de Aprendizaje

- [ ] Entiendo que regularizar = agregar un costo a la complejidad
- [ ] Puedo explicar por qué L1 produce coeficientes = 0
- [ ] Puedo explicar por qué L2 es más estable ante multicolinealidad
- [ ] Sé cuándo usar cada uno según los datos
- [ ] Entiendo el rol de λ y α
- [ ] Puedo implementar y comparar los tres métodos en Python

---

**Próximo tema:** Regularización implícita (Early Stopping, Subsample, Colsample)
